# Bronze

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from config import (
    RAW_ROOT,
    BRONZE_NAMESPACE,
    IMDB_FILES,
    MOVIELENS_FILES,
    MOVIELENS_SNAPSHOT_DATE,
    IMDB_SNAPSHOT_DATE,
    SNAPSHOT_DATES,
    INGESTION_METADATA_COLUMNS,
    IMDB_BRONZE_TABLES,
    snapshot_predicate,
)

In [ ]:
from pyspark.sql import functions as F

## Inspect raw files

In [ ]:
for file in IMDB_FILES:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", "\t")
        .csv(f"{RAW_ROOT}/imdb/{file}.tsv.gz")
    )

    display(df.limit(5))

In [ ]:
for file in MOVIELENS_FILES:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_ROOT}/movielens/{file}.csv")
    )

    display(df.limit(5))

## Ingest Bronze tables

### MovieLens

In [ ]:
base_path = f"{RAW_ROOT}/movielens"

movielens_bronze_dfs = {}

for name in MOVIELENS_FILES:
    path = f"{base_path}/{name}.csv"

    df = (
        spark.read
        .option("header", True)
        .csv(path)
        .withColumn("_source_file", F.lit(path))
        .withColumn("_source_dataset", F.lit("movielens_32m"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_snapshot_date", F.lit(MOVIELENS_SNAPSHOT_DATE))
    )

    movielens_bronze_dfs[f"movielens_{name}"] = df

### IMDb

In [ ]:
base_path = f"{RAW_ROOT}/imdb"

imdb_bronze_dfs = {}

for name in IMDB_FILES:
    path = f"{base_path}/{name}.tsv.gz"

    df = (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .csv(path)
        .withColumn("_source_file", F.lit(path))
        .withColumn("_source_dataset", F.lit("imdb"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_snapshot_date", F.lit(IMDB_SNAPSHOT_DATE))
    )

    table_name = name.replace(".", "_")
    imdb_bronze_dfs[f"imdb_{table_name}"] = df

## Persist Bronze tables

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_NAMESPACE}")

bronze_dfs = {
    **movielens_bronze_dfs,
    **imdb_bronze_dfs,
}
for table_name, bronze_df in bronze_dfs.items():
    target_table = f"{BRONZE_NAMESPACE}.{table_name}"
    snapshot_date = SNAPSHOT_DATES[table_name.split("_", 1)[0]]

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", snapshot_predicate(snapshot_date))
        .saveAsTable(target_table)
    )

## Data check

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)

results = []

for table_name in IMDB_BRONZE_TABLES:
    df = spark.table(table_name)
    # Exclude Bronze ingestion metadata.
    source_columns = [
        column
        for column in df.columns
        if column not in INGESTION_METADATA_COLUMNS
    ]

    # Calculate the row count and all \N counts in one scan.
    aggregations = [
        F.count(F.lit(1)).alias("_total_rows")
    ]

    aggregations.extend([
        F.sum(
            F.when(
                F.col(column).cast("string") == r"\N",
                1,
            ).otherwise(0)
        )
        .cast("long")
        .alias(column)
        for column in source_columns
    ])

    # An aggregate always produces one result row.
    counts = df.agg(*aggregations).collect()[0].asDict()

    total_rows = counts["_total_rows"]

    for column in source_columns:
        marker_count = counts[column] or 0

        marker_percentage = (
            round((marker_count / total_rows) * 100, 4)
            if total_rows > 0
            else 0.0
        )

        results.append((
            table_name,
            column,
            total_rows,
            marker_count,
            marker_percentage,
        ))

result_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("total_rows", LongType(), False),
    StructField("backslash_n_count", LongType(), False),
    StructField("backslash_n_percentage", DoubleType(), False),
])

bronze_null_marker_profile = spark.createDataFrame(
    results,
    schema=result_schema,
)

# Show every checked column.
display(
    bronze_null_marker_profile.orderBy(
        "table_name",
        "column_name",
    )
)

# Show only columns containing literal \N values.
display(
    bronze_null_marker_profile
    .filter(F.col("backslash_n_count") > 0)
    .orderBy(
        F.desc("backslash_n_count"),
        "table_name",
        "column_name",
    )
)